# PhaseGridUNet training and evaluation

Choose one action in the configuration and use Run All. `none` launches no training, inference or geometry evaluation. Inspection is independent of the selected action. Run in the grid-geometry uv environment; reserve the test split for final reporting.


In [ ]:
from pathlib import Path
import json
import platform
import torch
from grid_geometry.dataset.loader import PhaseDataset, inspect_dataset
from grid_geometry.phase.train import Config, train
from grid_geometry.phase.model import PhaseGridUNet, load_phase_weights
from grid_geometry.phase.evaluate import evaluate
from grid_geometry.phase.visualize import plot_history, plot_sample

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "src" / "grid_geometry").is_dir():
    raise RuntimeError("Open the notebook from the grid-geometry project root or notebooks directory")
DATASETS = PROJECT / "datasets"
RUNS = PROJECT / "runs"
print(f"Project: {PROJECT} | Python {platform.python_version()} | PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}")


## Editable configuration

Dataset names correspond to `DESTINATION = PROJECT / "datasets" / name` in the generation notebook. Choose fresh run names for new runs. Resume requires the original dataset and matching training settings; set `RESUME_PROFILE` to the original action. Device may change on resume, but workers and the other objective and sequence settings must match.


In [ ]:
ACTION = "none"  # "none", "smoke", "development", or "resume"
SMOKE_DATASET = "phase_training_smoke"
DEVELOPMENT_DATASET = "phase_development_v3"
SMOKE_RUN = "phase-training-smoke"
DEVELOPMENT_RUN = "phase-baseline-v1"
RESUME_DATASET = SMOKE_DATASET
RESUME_RUN = SMOKE_RUN
RESUME_PROFILE = "smoke"  # "smoke" or "development"
INSPECT_RUN = DEVELOPMENT_RUN
INSPECT_DATASET = DEVELOPMENT_DATASET

DEVICE = "auto"  # development/resume-development: auto, cpu, mps, cuda
WORKERS = 0
BATCH_SIZE = 8  # development; smoke always uses 8
DEVELOPMENT_EPOCHS = 30
RESUME_EPOCHS = 3  # total epoch limit, not additional epochs
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
SEED = 42  # development; smoke always uses 42
AMP = False  # development; smoke always disables AMP

RUN_GEOMETRY_EVALUATION = False
RUN_PREDICTION_VISUALIZATION = False
GEOMETRY_LIMIT = None  # None evaluates all validation samples
GEOMETRY_SEED = 42
EVALUATION_DEVICE = "cpu"
INSPECTION_CHECKPOINT = "best.pt"  # or "last.pt"
INSPECTION_SAMPLE_INDEX = 0  # used if no geometry failures are available

if ACTION not in {"none", "smoke", "development", "resume"}:
    raise ValueError(f"Unknown ACTION: {ACTION}")
if RESUME_PROFILE not in {"smoke", "development"}:
    raise ValueError(f"Unknown RESUME_PROFILE: {RESUME_PROFILE}")
for name in (SMOKE_DATASET, DEVELOPMENT_DATASET, RESUME_DATASET, INSPECT_DATASET,
             SMOKE_RUN, DEVELOPMENT_RUN, RESUME_RUN, INSPECT_RUN):
    if not name or name in {".", ".."} or Path(name).name != name:
        raise ValueError(f"Use a single dataset or run directory name: {name!r}")
if INSPECTION_CHECKPOINT not in {"best.pt", "last.pt"}:
    raise ValueError("INSPECTION_CHECKPOINT must be best.pt or last.pt")
print(f"Action: {ACTION}; datasets: {DATASETS}; runs: {RUNS}")


## Verify the selected training dataset

Before a training action, stream the existing integrity checks, including manifest coverage and split identity isolation. This can take time on a development dataset. With `none`, no dataset is required.


In [ ]:
if ACTION != "none":
    dataset_name = {"smoke": SMOKE_DATASET, "development": DEVELOPMENT_DATASET,
                    "resume": RESUME_DATASET}[ACTION]
    dataset_root = DATASETS / dataset_name
    print(f"Verifying {dataset_root}")
    verification = inspect_dataset(dataset_root)
    for split in ("train", "val"):
        PhaseDataset(dataset_root, split, require_nonempty=True)
    profile = RESUME_PROFILE if ACTION == "resume" else ACTION
    if profile == "smoke":
        expected = {"train": 40, "val": 12, "test": 12}
        config_data = json.loads((dataset_root / "config.json").read_text())
        if verification["split_counts"] != expected or tuple(config_data["input_size"]) != (384, 384):
            raise ValueError(f"Smoke dataset must have {expected} samples at 384 x 384")
    print(verification)


## Smoke training

The functional check uses two epochs, CPU, batch size 8, seed 42 and no AMP. It starts from scratch in a separate directory. The trainer refuses nonempty runs; this notebook also refuses an existing empty directory.


In [ ]:
if ACTION == "smoke":
    run_dir = RUNS / SMOKE_RUN
    if run_dir.exists():
        raise FileExistsError(f"Choose a fresh smoke run name: {run_dir}")
    config = Config(str(dataset_root), str(run_dir), batch_size=8, workers=WORKERS,
                    epochs=2, device="cpu", seed=42, amp=False,
                    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = train(config)


## Development training

The provisional baseline starts from scratch with AdamW and no scheduler; it never loads smoke weights. Configure device and workers for the host, and use a fresh run name.


In [ ]:
if ACTION == "development":
    run_dir = RUNS / DEVELOPMENT_RUN
    if run_dir.exists():
        raise FileExistsError(f"Choose a fresh development run name: {run_dir}")
    config = Config(str(dataset_root), str(run_dir), batch_size=BATCH_SIZE, workers=WORKERS,
                    epochs=DEVELOPMENT_EPOCHS, device=DEVICE, seed=SEED, amp=AMP,
                    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = train(config)


## Explicit resume

Resume restores the full state from `last.pt`. Set the total epoch limit higher than the completed epoch and retain the original dataset and settings. The trainer verifies compatibility and raises on a mismatch; there is no fallback to a new run.


In [ ]:
if ACTION == "resume":
    run_dir = RUNS / RESUME_RUN
    checkpoint_path = run_dir / "last.pt"
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Resume requires a resumable checkpoint: {checkpoint_path}")
    smoke = RESUME_PROFILE == "smoke"
    config = Config(str(dataset_root), str(run_dir), batch_size=8 if smoke else BATCH_SIZE,
                    workers=WORKERS, epochs=RESUME_EPOCHS, device="cpu" if smoke else DEVICE,
                    seed=42 if smoke else SEED, amp=False if smoke else AMP,
                    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = train(config, resume=True)


## Inspect an existing run

History is read from the selected checkpoint independently of training. Checkpoints must be trusted. Missing artifacts are reported without failing the default Run All. Enable geometry evaluation or prediction visualization explicitly; both use validation data.


In [ ]:
inspect_dir = RUNS / INSPECT_RUN
inspect_checkpoint = inspect_dir / INSPECTION_CHECKPOINT
inspection_state = None
if inspect_checkpoint.is_file():
    inspection_state = torch.load(inspect_checkpoint, map_location="cpu", weights_only=False)
    print(f"Checkpoint: {inspect_checkpoint}; epoch: {inspection_state['epoch']}")
    plot_history(inspection_state["history"])
else:
    print(f"No checkpoint at {inspect_checkpoint}; choose an existing inspection run.")


## Optional geometry evaluation and prediction inspection

The seeded validation subset and its sample IDs are recorded in the report. Visualization selects the first failure when available, otherwise the configured validation index. The decoder and validator do not change checkpoint selection.


In [ ]:
report = None
if RUN_GEOMETRY_EVALUATION:
    if inspection_state is None:
        print("Geometry evaluation skipped: checkpoint is missing.")
    else:
        report = evaluate(inspect_checkpoint, DATASETS / INSPECT_DATASET, split="val",
                          device=EVALUATION_DEVICE, limit=GEOMETRY_LIMIT, seed=GEOMETRY_SEED, global_geometry=True)
        print("Historical local decoder and validator:", report["geometry"])
        print("Global single-crop recognition:", report["global_geometry_single_crop"])
        failures = [r for r in report["records"] if (not r["positive"] and (r["local_accepted"] or r["global"]["accepted"])) or (r["positive"] and ((not r["local_accepted"] or r["local_correct"] is not True) if not r["complete"] else (not r["global"]["accepted"] or r["global"]["correct"] is not True)))]
        print("First failures:", failures[:5])


In [ ]:
if RUN_PREDICTION_VISUALIZATION:
    if inspection_state is None:
        print("Prediction visualization skipped: checkpoint is missing.")
    else:
        val_set = PhaseDataset(DATASETS / INSPECT_DATASET, "val", require_nonempty=True)
        model = PhaseGridUNet()
        load_phase_weights(model, inspection_state)
        if report is not None and failures:
            index = val_set.names.index(failures[0]["sample_id"])
        else:
            index = INSPECTION_SAMPLE_INDEX
        if not 0 <= index < len(val_set):
            raise IndexError(f"Validation sample index {index} outside 0..{len(val_set)-1}")
        plot_sample(model, val_set.load_sample(index), device="cpu")
